# 02 - Relational Aggregation, Feature Engineering, Leakage Audit

**Purpose:** Aggregate history tables with DuckDB SQL into one row per SK_ID_CURR, engineer ratios, document features and availability.

**Inputs:** data/interim/*.parquet, sql/*.sql

**Outputs / persisted artifacts:** data/processed/features_{train,test}.parquet, feature_documentation.csv, feature_availability.csv, train_test_consistency.json

**Dependencies:** Notebook 01

**Execution order:** 2 of 6

**Validation checks:** Unique SK_ID_CURR, no row multiplication, train/test column consistency

_All logic lives in `src/creditiq`; cells are safe to re-run (completed stages are skipped, artifacts live on Google Drive)._

In [ ]:
# --- Colab setup (safe to re-run) ---
import os, sys, subprocess
from pathlib import Path
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
ROOT = Path("/content/drive/MyDrive/CreditIQ") if Path("/content/drive/MyDrive").exists() else Path.cwd().parent
os.environ["CREDITIQ_ROOT"] = str(ROOT); sys.path.insert(0, str(ROOT / "src"))
if not (ROOT / "src").exists():
    raise SystemExit(f"Project not found at {ROOT}. Unzip CreditIQ into MyDrive/CreditIQ first.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=False)
from creditiq.config import get_paths, load_config, read_state
paths = get_paths(); cfg = load_config(paths)
print("Project root:", ROOT); print("Last completed stage:", read_state(paths).get("last_completed"))

In [ ]:
from creditiq import pipeline
pipeline.run("aggregate"); pipeline.run("features")

In [ ]:
import pandas as pd
tr = pd.read_parquet(paths.processed / "features_train.parquet"); print(tr.shape, "unique ids:", tr.SK_ID_CURR.is_unique)
display(pd.read_csv(paths.reports / "feature_documentation.csv")); display(pd.read_csv(paths.reports / "feature_availability.csv").head(30))

**Leakage note.** The dataset provides relative day offsets only. The audit checks consistency with the documented convention (<= 0 = before application) and neutralises future-dated values; it cannot prove temporal validity independently.